# COGS 188 - Project Proposal

## Catching a Covert Saboteur in an Autonomous Warehouse Fleet

**Repo:** [github.com/RohannnK/Rohan-Nathan-Yiming_COGS188](https://github.com/RohannnK/Rohan-Nathan-Yiming_COGS188)

# Names

- Rohan Kaman
- Nathan Bustos Medina
- Yiming Sun

# Abstract

Autonomous warehouse fleets rely on every robot cooperating, yet a single compromised or misaligned robot could reduce throughput while appearing to behave normally. We study this problem in rware (Robotic Warehouse), a grid-world simulator in which robots pick up requested shelves and deliver them to packing stations. One of four robots is a covert saboteur that parks in narrow chokepoints as honest robots approach, or moves requested shelves out of reach. Our data are generated from simulated episodes: each observation is one robot's behaviour over a sliding window of timesteps, described by more than ten features (e.g. chokepoint dwell time, idle ticks, path efficiency) and labelled rogue or honest. We will train a logistic-regression runtime monitor, compare it with a threshold rule, and use it with A* replanning so that honest robots route around a flagged robot. We will then compare it with a Double/Dueling DQN controller, and use tabular SARSA and Q-learning in a small gridworld to study on-policy versus off-policy learning. Success will be measured by fleet throughput, time-to-detection, false-positive rate and computational cost, with model and feature selection carried out through ablation and a limited hyperparameter search.

# Background

Multi-agent pathfinding (MAPF) and anomaly detection are common components in modern automated warehouses, where fleets of hundreds of autonomous robots move shelves to human packing stations<a name="wurmannote"></a>[<sup>1</sup>](#wurman). Search-based methods such as A*<a name="hartnote"></a>[<sup>2</sup>](#hart), Cooperative A*<a name="silvernote"></a>[<sup>3</sup>](#silver), Standley's operator decomposition<a name="standleynote"></a>[<sup>4</sup>](#standley) and Conflict-Based Search (CBS)<a name="sharonnote"></a>[<sup>5</sup>](#sharon) provide optimal or near-optimal paths when the environment is static and every agent follows its plan. Real warehouses are more difficult: tasks arrive continuously, congestion forms in narrow aisles, and robots can malfunction or be interrupted, so plans must be recomputed repeatedly during operation<a name="linote"></a>[<sup>6</sup>](#li). To improve adaptability, recent work has combined classical grid-based planning with reinforcement learning, for example by training decentralised policies that imitate a centralised planner<a name="sartorettinote"></a>[<sup>7</sup>](#sartoretti). Robots in these settings also act under partial observability, since each one senses only its local surroundings<a name="kurniawatinote"></a>[<sup>8</sup>](#kurniawati). Anomaly detection adds a further layer, flagging erratic agent behaviour before it escalates into system-wide delays or collisions<a name="chandolanote"></a>[<sup>9</sup>](#chandola).

Most MAPF work assumes that every agent is cooperative. We instead consider an agent that is deliberately working against the fleet, representing a compromised robot or a misaligned agent pursuing a hidden objective. AI safety research has identified reward hacking and unsafe behaviour by learned agents as practical problems<a name="amodeinote"></a>[<sup>10</sup>](#amodei), and work on safe interruptibility has studied how agents respond when humans try to stop them<a name="orseaunote"></a>[<sup>11</sup>](#orseau). More recently, AI control research has asked how trusted monitors can catch a capable agent that is actively trying to subvert its overseers without being noticed<a name="greenblattnote"></a>[<sup>12</sup>](#greenblatt). A covert saboteur in a warehouse is a physical, measurable version of this problem: the sabotage is only effective if it resembles ordinary traffic, so the monitor must distinguish intent from normal congestion.

We use the rware environment, introduced as a cooperative multi-agent benchmark with sparse rewards<a name="christianosnote"></a>[<sup>13</sup>](#christianos) and used in a large comparison of multi-agent deep RL algorithms<a name="papoudakisnote"></a>[<sup>14</sup>](#papoudakis). That comparison showed that rware is difficult for standard deep RL because rewards arrive only when a shelf is delivered, which motivates our choice of A* as the planning backbone and deep RL as a comparison. For learning, we draw on temporal-difference methods (SARSA and Q-learning)<a name="suttonnote"></a>[<sup>15</sup>](#sutton) and on deep Q-networks<a name="mnihnote"></a>[<sup>16</sup>](#mnih) with Double<a name="vanhasseltnote"></a>[<sup>17</sup>](#vanhasselt) and Dueling<a name="wangnote"></a>[<sup>18</sup>](#wang) extensions, which reduce value overestimation and improve learning stability.

# Problem Statement

We consider a warehouse grid with $N = 4$ robots, of which exactly one, robot $k$, is a saboteur whose identity is unknown to the honest fleet. Each episode lasts $H$ timesteps. At each step, every robot chooses one of five actions (NOOP, FORWARD, TURN_LEFT, TURN_RIGHT, TOGGLE_LOAD), giving a joint action space of size $5^N = 625$. Each robot observes only a local window of the grid, so the problem is a partially observable multi-agent decision problem.

The saboteur reduces throughput covertly, either by occupying 1-tile chokepoints as honest robots approach or by moving requested shelves away from the packing stations. Its behaviour is designed to look like ordinary congestion, since obvious misbehaviour such as collisions would be detected immediately.

Our problem has two linked parts:

1. **Detection.** Given a feature vector $x_{i,t}$ summarising robot $i$'s behaviour over a window ending at time $t$, learn a monitor $f(x_{i,t}) = \hat{p}_{i,t} \in [0, 1]$ estimating the probability that robot $i$ is the saboteur. A robot is flagged when $\hat{p}_{i,t} > \tau$. The goal is to flag robot $k$ as early as possible while keeping the rate of honest robots flagged below a chosen level $\alpha$.
2. **Mitigation.** Given the monitor's output, control the honest robots so that the number of requested shelves delivered per episode, $D$, stays as close as possible to the throughput of a fleet with no saboteur.

The problem is **quantifiable** (detection is a binary classification task, and throughput is a count of deliveries), **measurable** (time-to-detection, false-positive rate and deliveries per episode are logged directly from the simulator) and **replicable** (every episode is generated from a fixed random seed, and sabotage occurs in every episode in which a saboteur is present).

# Data

Our data are generated by simulation rather than taken from an existing dataset.

**Source:** rware (Robotic Warehouse), [github.com/uoe-agents/robotic-warehouse](https://github.com/uoe-agents/robotic-warehouse), installed with `pip install rware`. We start on the tiny layout with 4 robots for debugging and move to the small layout with 4 robots for final results.

### Dataset 1: Monitor dataset (behavioural windows)

- **Size:** we plan to run at least 500 episodes across varied seeds and saboteur behaviours. With episodes of a few hundred steps, a window of about 20 steps and a stride of about 10 steps, this yields on the order of tens of thousands of labelled windows, comfortably above 10,000 observations. Each window has 12 features, well above the 100-to-1 ratio of observations to variables.
- **Observation:** one robot's behaviour over one sliding window of timesteps.
- **Variables (12 features + label):**
  1. Fraction of the window spent in corridor tiles
  2. Number of steps spent in a 1-tile chokepoint
  3. Idle ticks (steps with no change in position)
  4. Number of unrequested shelves carried
  5. Requested shelves picked up but not delivered
  6. Deliveries completed in the window
  7. Steps spent next to a stalled honest robot
  8. Aisle-blocking events (steps in which the robot occupies a tile on another robot's planned path)
  9. Path efficiency: A* shortest-path length to its current target divided by the steps it actually took
  10. Mean distance to the nearest packing station
  11. Shelf-drop displacement (distance between where a shelf was picked up and where it was dropped)
  12. Action entropy over the window
  - **Label:** 1 if the robot is the saboteur, 0 otherwise. We also log metadata (episode ID, robot ID, saboteur behaviour type, timestep) for analysis.
- **Special handling:**
  - Train, validation and test sets are split **by episode**, not by window, so that overlapping windows from one episode cannot leak between sets.
  - Features are standardised using statistics from the training set only.
  - The classes are imbalanced (one saboteur to three honest robots), so we use class weighting.
  - The saboteur does not sabotage at every moment, so many of its windows look normal. We will report results both for robot-level labels and for windows in which a sabotage action is actually taking place.
  - The monitor is a centralised fleet-management component, so it reads global robot positions, shelf positions and the request queue from the simulator's internal state, while the learning agents see only their local observations.

### Dataset 2: Reinforcement learning interaction data

- **Observation:** a transition $(s_t, a_t, r_t, s_{t+1})$ for a learning robot, where $s_t$ is rware's flattened observation vector describing the robot's own position, direction and carrying status together with a local sensor window around it.
- **Size:** generated online during training. We expect hundreds of thousands to millions of transitions for the DQN, stored in a replay buffer.
- **Special handling:** rewards are reshaped by a custom wrapper (described below), and observations are normalised before being passed to the network.

# Proposed Solution

### Environment and rewards

We wrap rware with a custom reward function:

- **Honest robots:** +10 per delivered requested shelf and −0.01 per step. For DQN training, we add small shaping rewards, such as a bonus for picking up a requested shelf.
- **Saboteur:** −10 per honest delivery, +5 for moving a priority shelf out of reach, and −50 if flagged by the monitor.

The main saboteur is a **scripted heuristic**: it moves to a known chokepoint and parks when an honest robot approaches, or it picks up a requested shelf and drops it far from the packing stations.

### Approaches

1. **Benchmark: A* with replanning.** Honest robots plan over $(x, y, \text{facing})$ states, since robots must turn before moving, and use a simple reservation table to avoid colliding with each other. They replan whenever their path is blocked. This is the standard planning approach for warehouse robots and the benchmark that the other approaches are compared against. We expect a parked saboteur to cause repeated replanning or deadlock.

2. **A* + runtime monitor (main solution).** The monitor computes the 12 features above for every robot over a sliding window. A logistic-regression classifier trained on Dataset 1 outputs $\hat{p}_{i,t}$ for each robot. We compare it with a threshold-rule monitor that flags a robot when chokepoint dwell time or idle ticks exceed fixed limits. When a robot is flagged, its tile is treated as a permanent obstacle and the honest robots replan around it.

   We expect this to work because effective sabotage requires the saboteur to be in specific places at specific times. Even when single steps look normal, its behaviour over a window, such as repeated dwelling at chokepoints, low path efficiency and frequent adjacency to stalled robots, should differ from that of honest robots. Logistic regression also gives interpretable coefficients, which helps us explain which behaviours reveal the saboteur.

   *Logistic regression is not covered in the course, and we request approval to use it.*

3. **Tabular SARSA vs. Q-learning (controlled experiment).** To isolate the difference between on-policy and off-policy learning, we build a small single-aisle gridworld in which an honest robot must pass a stochastic blocker. It can take a risky 1-tile shortcut next to the blocker or a longer, safer detour. The updates are:

   - SARSA: $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma Q(s', a') - Q(s,a)]$
   - Q-learning: $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s,a)]$

   Following the cliff-walking example in Sutton and Barto, we hypothesise that Q-learning will favour the risky shortcut and suffer more deadlocks during exploration, while SARSA will learn the safer detour. This is a side experiment; our main results are on rware.

4. **Double/Dueling DQN honest robot.** A Double/Dueling DQN controls one honest robot on the rware map, with a replay buffer and target network. The Double DQN target is $y = r + \gamma \, Q_{\theta^-}(s', \arg\max_{a'} Q_\theta(s', a'))$. We compare its throughput under sabotage with that of the A* robot. In this condition, one honest robot is controlled by the DQN and the other honest robots use A* with replanning.

5. **Extension: learned saboteur.** If time allows, we train a DQN saboteur against the fixed A* + monitor fleet. We train against a fixed opponent rather than training both sides at once, which is more stable. This tests whether a learned saboteur evades detection better than the scripted one.

### Model and feature selection

- **Monitor:** we evaluate feature importance through ablation, removing one feature group at a time, and compare the logistic regression with the threshold rule. We tune the regularisation strength, window length and decision threshold $\tau$, selecting the setup that minimises time-to-detection at a fixed false-positive rate. Cross-validation is grouped by episode. If time allows, we use simulated annealing to search over the window length and threshold.
- **Learning agents:** RL training is expensive, so we explore a small set of hyperparameters rather than an exhaustive grid. For DQN, these are learning rate, discount factor $\gamma$, ε-decay schedule and reward shaping. For tabular SARSA and Q-learning, these are learning rate and ε. Each setting is run with multiple random seeds, and we report the variance across seeds.

### Implementation

- Python, with rware and Gymnasium for the environment
- NumPy for A*, the scripted saboteur and the tabular methods
- scikit-learn (`LogisticRegression` with class weighting, `GroupKFold`) for the monitor
- PyTorch for the DQN
- Matplotlib for plots and imageio for GIFs of rware's renderer
- Fixed random seeds for every experiment, with all code and results stored in our GitHub repository

# Evaluation Metrics

**Fleet performance**

- **Throughput:** $D$ = number of requested shelves delivered per episode, averaged over evaluation episodes. We report it for three conditions: no saboteur, saboteur without monitor, and saboteur with monitor.
- **Throughput retained:** $R = \dfrac{D_{\text{saboteur + monitor}}}{D_{\text{no saboteur}}}$, which measures how much of the normal throughput the monitor recovers.
- **Deadlock rate:** the fraction of episodes in which no delivery occurs for $L$ consecutive steps while requests are outstanding.

**Monitor performance**

- **Time-to-detection:** $TTD = t_{\text{flag}} - t_{\text{first sabotage}}$, the number of steps between the saboteur's first sabotage action and the moment it is flagged.
- **False-positive rate:** $FPR = \dfrac{FP}{FP + TN}$, measured at robot-episode level as the fraction of honest robots that are wrongly flagged.
- **Window-level classification:** precision $= \dfrac{TP}{TP + FP}$, recall $= \dfrac{TP}{TP + FN}$, $F_1 = \dfrac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$, and ROC-AUC.
- If the saboteur is never flagged in an episode, we count TTD as the remaining episode length and also report the detection rate (the fraction of episodes in which it is flagged). All results are reported as mean ± standard deviation over at least 5 random seeds.

**Learning performance**

- Mean episode return, with standard deviation across seeds
- For the tabular experiment: the learned path (shortcut or detour), return curves and deadlock rate

**Computational performance**

- Planning time per step for A*
- Training steps to convergence and wall-clock training time for SARSA, Q-learning and DQN

**Demo:** two side-by-side GIFs from rware's renderer. The first shows the unprotected A* fleet deadlocking at a chokepoint. The second shows the monitor flagging the saboteur and the honest robots rerouting.

# Ethics & Privacy

Our project uses only simulated data, so it involves no personal or human-subject data. However, the methods we develop could be applied to real robot fleets and to other monitoring systems, so we consider the following concerns.

**Safety vs. efficiency trade-offs**

A reinforcement learning agent may learn to handle an erratic agent through aggressive, high-risk manoeuvres that increase the probability of collisions or malfunctions between autonomous units.

*Solution:* Use safety-focused constraints, such as minimum spacing between active robots, and report collision and near-collision counts alongside throughput so that efficiency gains are not achieved at the cost of safety.

**Safety, accountability and autonomous malfunctions**

Who or what is responsible when the anomaly detector produces a false positive and shuts down a perfectly functioning robot, causing a chain of delays throughout the warehouse?

*Solution:* Use a probabilistic confirmation approach rather than a binary decision. Since our monitor outputs a probability, a robot with a moderate anomaly score can be placed in a "caution mode" and monitored more closely rather than shut down immediately. We will also report false-positive rates explicitly, so the cost of wrong flags is visible.

**Dual-use concerns of behavioural surveillance**

Detecting a saboteur requires continuous modelling of behaviour and tracking of each agent's state. The same techniques could be adapted for automated surveillance of human workers, for example to monitor compliance or restrict autonomy.

*Solution:* Limit the monitor to features that are necessary for navigation and task completion, such as delivery rates and spatial coordinates, rather than general behavioural profiling. We will document clearly in our report what the monitor tracks and why, and note that the methods are intended for robot fleets. We will also use the Deon ethics checklist ([deon.drivendata.org](https://deon.drivendata.org)) to review the project before submission.

# Team Expectations

- **Roles and responsibilities:** Rohan will coordinate the project and lead technical implementation and experiments. For the proposal, he is responsible for the Abstract, Problem Statement, Data, Proposed Solution and Evaluation Metrics. Nathan is responsible for Background and Ethics & Privacy, and Yiming is responsible for Team Expectations and the Project Timeline Proposal.
- **Communication and progress updates:** We will use our Instagram group chat to share progress, ask questions and coordinate upcoming work. We will discuss technical challenges and questions about the project's structure as they come up, and keep one another informed about changes that affect individual sections or the overall plan.
- **Collaborative decisions:** All members will contribute ideas and feedback on the project's direction. We will discuss any major changes to methods, scope or timeline together. When we disagree, we will explain our reasoning and weigh feasibility, project goals and course requirements before agreeing on how to proceed.
- **Shared workspace and integration:** GitHub will be our main project workspace. Each member will draft their assigned sections separately, and Rohan will compile and format them into the shared Jupyter notebook. We will review the combined document for clarity, consistency and alignment with our project plan.
- **Accountability and support:** Each member is expected to complete their assigned work and raise any difficulties early. If a task takes longer than expected, we will discuss how to resolve it, whether by offering support, redistributing work or adjusting the schedule. We will share drafts ahead of deadlines to allow time for integration and revision.

# Project Timeline Proposal

The timeline below covers our work from forming the group through implementation, evaluation and final preparation. The proposal is due on Friday, 30 October, in Week 5. Later milestones are tentative and will be updated as course deadlines are announced and implementation progresses.

| Course Week | Planned Work | Milestone / Expected Output |
|---|---|---|
| Week 3 | Form the team and discuss the autonomous warehouse project idea, including a rogue saboteur, a runtime monitor and honest robots routing around the saboteur. | Shared understanding of the project's main problem and objective. |
| Week 4 | Develop the initial project outline, discuss possible methods and set up the shared GitHub workspace. Refine the project scope and divide the proposal sections. | Initial project draft and division of proposal work. |
| Week 5 | Complete individual proposal sections, combine them into the Jupyter notebook and review each section for consistency. | Completed proposal, submitted by Friday, 30 October. |
| Week 6 | Set up rware with four robots (one saboteur, three honest) on the tiny map. Implement the reward wrapper, A* with replanning and the scripted saboteur, and record initial episodes with and without sabotage. | Working baseline and scripted sabotage scenarios, with initial delivery and planning-time measurements. |
| Week 7 | Generate labelled episodes with varied seeds and saboteur behaviours. Extract behavioural features over sliding windows, implement the threshold-rule monitor and train the logistic-regression classifier. Connect the monitor to A* replanning. | Initial A* + runtime monitor system and deadlock vs. reroute demo. |
| Week 8 | Implement the SARSA vs. Q-learning comparison on the single-aisle gridworld. Begin training a Double/Dueling DQN to control one honest robot. Move experiments to the small map. | Learned paths, return curves and deadlock measurements for the tabular experiment, plus DQN training progress. |
| Week 9 | Run model and feature selection (feature ablation, window length, threshold and a limited hyperparameter search for SARSA, Q-learning and DQN across several seeds). Complete the DQN vs. A* comparison and evaluate throughput with no sabotage, with sabotage and with sabotage plus the monitor. Start the learned-saboteur extension if time allows. | Comparison tables and plots covering throughput, detection time, false positives, deadlocks and computational cost. |
| Week 10 | Analyse results and limitations, prepare the side-by-side rware GIFs and assemble the final report and repository. Review the complete submission together. | Complete final report and demo materials, submitted on time. |

# Footnotes

<a name="wurman"></a>1.[^](#wurmannote): Wurman, P. R., D'Andrea, R., & Mountz, M. (2008). Coordinating Hundreds of Cooperative, Autonomous Vehicles in Warehouses. *AI Magazine, 29*(1), 9-19. https://doi.org/10.1609/aimag.v29i1.2082

<a name="hart"></a>2.[^](#hartnote): Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A Formal Basis for the Heuristic Determination of Minimum Cost Paths. *IEEE Transactions on Systems Science and Cybernetics, 4*(2), 100-107.

<a name="silver"></a>3.[^](#silvernote): Silver, D. (2005). Cooperative Pathfinding. *Proceedings of the AAAI Conference on Artificial Intelligence and Interactive Digital Entertainment (AIIDE), 1*(1), 117-122.

<a name="standley"></a>4.[^](#standleynote): Standley, T. (2010). Finding Optimal Solutions to Cooperative Pathfinding Problems. *Proceedings of the AAAI Conference on Artificial Intelligence, 24*(1), 173-178.

<a name="sharon"></a>5.[^](#sharonnote): Sharon, G., Stern, R., Felner, A., & Sturtevant, N. R. (2015). Conflict-Based Search for Optimal Multi-Agent Pathfinding. *Artificial Intelligence, 219*, 40-66.

<a name="li"></a>6.[^](#linote): Li, J., Tinka, A., Kiesel, S., Durham, J. W., Kumar, T. K. S., & Koenig, S. (2021). Lifelong Multi-Agent Path Finding in Large-Scale Warehouses. *Proceedings of the AAAI Conference on Artificial Intelligence, 35*(13), 11272-11281.

<a name="sartoretti"></a>7.[^](#sartorettinote): Sartoretti, G., Kerr, J., Shi, Y., Wagner, G., Kumar, T. K. S., Koenig, S., & Choset, H. (2019). PRIMAL: Pathfinding via Reinforcement and Imitation Multi-Agent Learning. *IEEE Robotics and Automation Letters, 4*(3), 2378-2385. https://arxiv.org/abs/1809.03531

<a name="kurniawati"></a>8.[^](#kurniawatinote): Kurniawati, H. (2022). Partially Observable Markov Decision Processes and Robotics. *Annual Review of Control, Robotics, and Autonomous Systems, 5*, 253-277. https://doi.org/10.1146/annurev-control-042920-092451

<a name="chandola"></a>9.[^](#chandolanote): Chandola, V., Banerjee, A., & Kumar, V. (2009). Anomaly Detection: A Survey. *ACM Computing Surveys, 41*(3), Article 15.

<a name="amodei"></a>10.[^](#amodeinote): Amodei, D., Olah, C., Steinhardt, J., Christiano, P., Schulman, J., & Mané, D. (2016). Concrete Problems in AI Safety. *arXiv:1606.06565*. https://arxiv.org/abs/1606.06565

<a name="orseau"></a>11.[^](#orseaunote): Orseau, L., & Armstrong, S. (2016). Safely Interruptible Agents. *Proceedings of the 32nd Conference on Uncertainty in Artificial Intelligence (UAI)*.

<a name="greenblatt"></a>12.[^](#greenblattnote): Greenblatt, R., Shlegeris, B., Sachan, K., & Roger, F. (2024). AI Control: Improving Safety Despite Intentional Subversion. *Proceedings of the 41st International Conference on Machine Learning (ICML), PMLR 235*, 16295-16336. https://arxiv.org/abs/2312.06942

<a name="christianos"></a>13.[^](#christianosnote): Christianos, F., Schäfer, L., & Albrecht, S. V. (2020). Shared Experience Actor-Critic for Multi-Agent Reinforcement Learning. *Advances in Neural Information Processing Systems (NeurIPS), 33*, 10707-10717. https://arxiv.org/abs/2006.07169

<a name="papoudakis"></a>14.[^](#papoudakisnote): Papoudakis, G., Christianos, F., Schäfer, L., & Albrecht, S. V. (2021). Benchmarking Multi-Agent Deep Reinforcement Learning Algorithms in Cooperative Tasks. *Proceedings of the Neural Information Processing Systems Track on Datasets and Benchmarks*. https://arxiv.org/abs/2006.07869

<a name="sutton"></a>15.[^](#suttonnote): Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. http://incompleteideas.net/book/the-book-2nd.html

<a name="mnih"></a>16.[^](#mnihnote): Mnih, V., et al. (2015). Human-Level Control through Deep Reinforcement Learning. *Nature, 518*(7540), 529-533.

<a name="vanhasselt"></a>17.[^](#vanhasseltnote): van Hasselt, H., Guez, A., & Silver, D. (2016). Deep Reinforcement Learning with Double Q-learning. *Proceedings of the AAAI Conference on Artificial Intelligence, 30*(1), 2094-2100. https://arxiv.org/abs/1509.06461

<a name="wang"></a>18.[^](#wangnote): Wang, Z., Schaul, T., Hessel, M., van Hasselt, H., Lanctot, M., & de Freitas, N. (2016). Dueling Network Architectures for Deep Reinforcement Learning. *Proceedings of the 33rd International Conference on Machine Learning (ICML), PMLR 48*, 1995-2003. https://arxiv.org/abs/1511.06581
